# 03 — Train and evaluate

A station/month/hour climatology shrunk toward global training prevalence is
the baseline. The model comparison uses 2017–2022 for training, Jan–Jun 2023 for selection,
Jul–Dec 2023 for optional probability calibration, and 2024 as the final holdout.
The selection policy ranks mean PR-AUC over supported thresholds, uses Brier score
to break close ties, and reports F1 at a fixed 0.5 reference cutoff. No cutoff is
optimized. Run the final evaluation cell only after the selection and calibration
cells have completed.


In [ ]:
import json
import math
import os
from decimal import Decimal
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display
from pyspark import StorageLevel
from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import Imputer, OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.regression import IsotonicRegression
from pyspark.sql import SparkSession, Window, functions as F


In [ ]:
BUCKET = os.environ.get("BUCKET", "sg-rainfall-dsa5208-261011-data")
RUN_ID = os.environ.get("RUN_ID", "20261012-full-01")
ROOT = f"gs://{BUCKET}/runs/{RUN_ID}"
MODEL_ROOT = f"{ROOT}/models"
RESULTS_ROOT = f"{ROOT}/results"
config_result = subprocess.run(
    ["gcloud", "storage", "cat", f"{ROOT}/config/experiment.json"],
    check=True, capture_output=True, text=True,
)
config = json.loads(config_result.stdout)

spark = SparkSession.builder.appName("dsa5208-rainfall-models").getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", str(config["shuffle_partitions"]))
spark.conf.set("spark.sql.session.timeZone", config["timezone"])
spark.conf.set("spark.sql.ansi.enabled", "false")
actual_spark_conf = {
    name: spark.conf.get(name)
    for name in ("spark.sql.shuffle.partitions", "spark.sql.session.timeZone", "spark.sql.ansi.enabled")
}
assert actual_spark_conf["spark.sql.shuffle.partitions"] == str(config["shuffle_partitions"])
assert actual_spark_conf["spark.sql.session.timeZone"] == config["timezone"]
assert actual_spark_conf["spark.sql.ansi.enabled"] == "false"
display({"actual_spark_conf": actual_spark_conf})

assert config["availability_mode"] == "retrospective_event_time"
assert config["alignment_tolerance_seconds"] == 0
assert config["interval_semantics_confirmed"] or config["interval_end_assumption_accepted"]


In [ ]:
BASE_FEATURES = ["hour_sin", "hour_cos", "day_sin", "day_cos", "threshold_mm"]

def feature_columns(variant):
    if variant == "A":
        return list(BASE_FEATURES)
    if variant != "B":
        raise ValueError("Feature variant must be A or B.")
    columns = BASE_FEATURES + ["rain_current"]
    for minutes in config["lag_minutes"]:
        columns += [f"lag_{minutes}", f"lag_{minutes}_missing"]
    for minutes in config["rolling_minutes"]:
        columns += [
            f"sum_{minutes}", f"max_{minutes}", f"wet_count_{minutes}",
            f"count_{minutes}", f"coverage_{minutes}",
        ]
    return columns


In [ ]:
def threshold_units(value):
    scaled = Decimal(str(value)) * config["rainfall_integer_scale"]
    if scaled != scaled.to_integral_value():
        raise ValueError(f"Threshold {value} exceeds the stored numeric precision.")
    return int(scaled)

def expand_thresholds(frame, thresholds=None):
    values = thresholds or config["thresholds_mm"]
    rows = [(float(value), threshold_units(value)) for value in values]
    grid = spark.createDataFrame(rows, "threshold_mm double, threshold_units long")
    expanded = frame.crossJoin(F.broadcast(grid))
    return expanded.withColumn(
        "label", (F.col("future_units") > F.col("threshold_units")).cast("double")
    )


In [ ]:
def deterministic_sample(frame, fraction):
    if fraction >= 1:
        return frame
    bucket = F.pmod(
        F.xxhash64("station_id", "epoch", F.lit(config["sampling_seed"])),
        F.lit(1_000_000),
    )
    return frame.where(bucket < int(fraction * 1_000_000))

def mark_unseen(frame, seen_stations):
    return (
        frame.join(seen_stations.withColumn("_seen", F.lit(True)), "station_id", "left")
        .withColumn("unseen_station", F.col("_seen").isNull())
        .drop("_seen")
    )


In [ ]:
base = spark.read.parquet(f"{ROOT}/base_features")
train_base = base.where("split = 'train' AND label_valid")
selection_base = base.where("split = 'selection' AND label_valid")
calibration_base = base.where("split = 'calibration' AND label_valid")

for name, frame in [
    ("train", train_base),
    ("selection", selection_base),
    ("calibration", calibration_base),
]:
    if frame.limit(1).count() == 0:
        raise ValueError(f"The {name} split has no valid six-reading targets.")


In [ ]:
training_stations = train_base.select("station_id").distinct().persist()
threshold_values = config["thresholds_mm"]
support_counts = train_base.agg(
    F.count("*").alias("n"),
    *[
        F.sum(
            (F.col("future_units") > F.lit(threshold_units(value))).cast("long")
        ).alias(f"positive_{index}")
        for index, value in enumerate(threshold_values)
    ],
).first().asDict()
support = [
    {
        "threshold_mm": float(value),
        "n": int(support_counts["n"]),
        "positives": int(support_counts[f"positive_{index}"] or 0),
    }
    for index, value in enumerate(threshold_values)
]
primary_thresholds = sorted(
    row["threshold_mm"]
    for row in support
    if config["primary_threshold_min_positives"] <= row["positives"] < row["n"]
)
if not primary_thresholds:
    raise ValueError("No threshold has sufficient positive and negative training support.")
display(support)


In [ ]:
pilot = expand_thresholds(
    deterministic_sample(train_base, config["pilot_fraction"])
).persist(StorageLevel.DISK_ONLY)
train = expand_thresholds(
    deterministic_sample(train_base, config["final_train_fraction"])
).persist(StorageLevel.DISK_ONLY)
selection = mark_unseen(expand_thresholds(selection_base), training_stations).persist(
    StorageLevel.DISK_ONLY
)


In [ ]:
def histogram(keys):
    return (
        train_base.groupBy(*keys, "future_units")
        .count()
        .withColumnRenamed("count", "frequency")
    )

station_hist = histogram(["station_id", "month", "hour"])
month_hist = histogram(["month", "hour"])
global_hist = histogram([])
climatology_root = f"{MODEL_ROOT}/climatology"
station_hist.write.mode("overwrite").parquet(f"{climatology_root}/station_hour")
month_hist.write.mode("overwrite").parquet(f"{climatology_root}/month_hour")
global_hist.write.mode("overwrite").parquet(f"{climatology_root}/global")

display({
    "pilot_base_rows": pilot.select("station_id", "epoch").distinct().count(),
    "final_train_base_rows": train.select("station_id", "epoch").distinct().count(),
    "selection_rows_after_threshold_expansion": selection.count(),
    "primary_thresholds_mm": primary_thresholds,
})


In [ ]:
def global_climatology_probability(query):
    threshold_grid = query.select("threshold_mm", "threshold_units").distinct()
    return (
        threshold_grid.crossJoin(global_hist)
        .groupBy("threshold_mm", "threshold_units")
        .agg(
            F.sum("frequency").alias("global_n"),
            F.sum(
                F.when(
                    F.col("future_units") > F.col("threshold_units"),
                    F.col("frequency"),
                ).otherwise(0)
            ).alias("global_positives"),
        )
        .withColumn("global_p", F.col("global_positives") / F.col("global_n"))
    )


In [ ]:
def month_hour_climatology_probability(query, global_p):
    smoothing = config["climatology_smoothing"]
    return (
        query.select("month", "hour", "threshold_mm", "threshold_units")
        .distinct()
        .join(month_hist, ["month", "hour"], "left")
        .groupBy("month", "hour", "threshold_mm", "threshold_units")
        .agg(
            F.coalesce(F.sum("frequency"), F.lit(0)).alias("broad_n"),
            F.coalesce(
                F.sum(
                    F.when(
                        F.col("future_units") > F.col("threshold_units"),
                        F.col("frequency"),
                    ).otherwise(0)
                ),
                F.lit(0),
            ).alias("broad_positives"),
        )
        .join(
            global_p.select("threshold_mm", "threshold_units", "global_p"),
            ["threshold_mm", "threshold_units"],
        )
        .withColumn(
            "broad_p",
            (F.col("broad_positives") + smoothing * F.col("global_p"))
            / (F.col("broad_n") + smoothing),
        )
    )


In [ ]:
def station_hour_climatology_probability(query, broad):
    keys = ["station_id", "month", "hour", "threshold_mm", "threshold_units"]
    smoothing = config["climatology_smoothing"]
    return (
        query.join(station_hist, ["station_id", "month", "hour"], "left")
        .groupBy(*keys)
        .agg(
            F.coalesce(F.sum("frequency"), F.lit(0)).alias("local_n"),
            F.coalesce(
                F.sum(
                    F.when(
                        F.col("future_units") > F.col("threshold_units"),
                        F.col("frequency"),
                    ).otherwise(0)
                ),
                F.lit(0),
            ).alias("local_positives"),
        )
        .join(
            broad.select(
                "month", "hour", "threshold_mm", "threshold_units", "broad_p"
            ),
            ["month", "hour", "threshold_mm", "threshold_units"],
        )
        .withColumn(
            "p",
            (F.col("local_positives") + smoothing * F.col("broad_p"))
            / (F.col("local_n") + smoothing),
        )
    )


In [ ]:
def climatology_scores(queries):
    keys = ["station_id", "month", "hour", "threshold_mm", "threshold_units"]
    query = queries.select(*keys).distinct()
    global_p = global_climatology_probability(query)
    broad = month_hour_climatology_probability(query, global_p)
    local = station_hour_climatology_probability(query, broad)
    return queries.join(local.select(*keys, "p"), keys, "left")


In [ ]:
def feature_preprocessing_stages(variant):
    numeric = feature_columns(variant)
    imputed = [f"{name}_imputed" for name in numeric]
    return [
        StringIndexer(inputCol="station_id", outputCol="station_index", handleInvalid="keep"),
        OneHotEncoder(
            inputCols=["station_index"], outputCols=["station_vector"],
            handleInvalid="keep", dropLast=False,
        ),
        Imputer(inputCols=numeric, outputCols=imputed, strategy="median"),
        VectorAssembler(inputCols=["station_vector", *imputed], outputCol="assembled"),
    ]


In [ ]:
def build_pipeline(algorithm, parameters, variant):
    stages = feature_preprocessing_stages(variant)
    if algorithm == "lr":
        stages += [
            StandardScaler(
                inputCol="assembled", outputCol="features",
                withMean=False, withStd=True,
            ),
            LogisticRegression(
                featuresCol="features", labelCol="label", maxIter=50,
                elasticNetParam=0.0, **parameters,
            ),
        ]
    elif algorithm == "gbt":
        stages += [
            VectorAssembler(inputCols=["assembled"], outputCol="features"),
            GBTClassifier(
                featuresCol="features", labelCol="label",
                seed=config["sampling_seed"], maxBins=64, **parameters,
            ),
        ]
    else:
        raise ValueError(f"Unknown algorithm: {algorithm}")
    return Pipeline(stages=stages)


In [ ]:
def score_model(model, frame):
    return model.transform(frame).withColumn("p", vector_to_array("probability")[1])


In [ ]:
REFERENCE_CUTOFF = 0.5

def add_reference_classification_metrics(row):
    positives = int(row["positives"] or 0)
    precision = row["tp"] / (row["tp"] + row["fp"]) if row["tp"] + row["fp"] else None
    recall = row["tp"] / (row["tp"] + row["fn"]) if row["tp"] + row["fn"] else None
    row.update(
        positives=positives,
        prevalence=positives / row["n"] if row["n"] else None,
        reference_cutoff=REFERENCE_CUTOFF,
        precision=precision,
        recall=recall,
        f1=(
            2 * precision * recall / (precision + recall)
            if precision is not None and recall is not None and precision + recall
            else None
        ),
    )
    return row


In [ ]:
def probability_metrics(scores):
    p = F.col("p")
    y = F.col("label")
    row = scores.agg(
        F.count("*").alias("n"),
        F.sum(y).alias("positives"),
        F.avg(F.pow(p - y, 2)).alias("brier"),
        F.sum(((p >= REFERENCE_CUTOFF) & (y == 1)).cast("long")).alias("tp"),
        F.sum(((p >= REFERENCE_CUTOFF) & (y == 0)).cast("long")).alias("fp"),
        F.sum(((p < REFERENCE_CUTOFF) & (y == 1)).cast("long")).alias("fn"),
    ).first().asDict()

    positives = int(row["positives"] or 0)
    n = row["n"]
    row["pr_auc"] = None
    if 0 < positives < n:
        evaluator = BinaryClassificationEvaluator(
            rawPredictionCol="p", labelCol="label", metricName="areaUnderPR",
            numBins=config["auc_num_bins"],
        )
        row["pr_auc"] = evaluator.evaluate(scores)
    return add_reference_classification_metrics(row)


In [ ]:
def threshold_metrics(scores, thresholds):
    return [
        {"threshold_mm": float(value), **probability_metrics(
            scores.where(F.col("threshold_mm") == float(value))
        )}
        for value in thresholds
    ]


In [ ]:
def reliability_table(scores):
    bins = config["calibration_bins"]
    return (
        scores.withColumn(
            "bin", F.least(F.floor(F.col("p") * bins), F.lit(bins - 1)).cast("int")
        )
        .groupBy("threshold_mm", "bin")
        .agg(
            F.avg("p").alias("predicted_probability"),
            F.avg("label").alias("observed_frequency"),
            F.count("*").alias("n"),
        )
    )

def weighted_ece(scores):
    bins = reliability_table(scores)
    return bins.withColumn(
        "weighted_error",
        F.abs(F.col("predicted_probability") - F.col("observed_frequency")) * F.col("n"),
    ).agg((F.sum("weighted_error") / F.sum("n")).alias("ece")).first()["ece"]



In [ ]:
def selection_objective(record):
    supported = [
        row for row in record["metrics"]
        if row["threshold_mm"] in primary_thresholds and row["pr_auc"] is not None
    ]
    if not supported:
        return None, math.inf
    return (
        sum(row["pr_auc"] for row in supported) / len(supported),
        sum(row["brier"] for row in supported) / len(supported),
    )


In [ ]:
def choose_candidate(records):
    ranked = []
    for record in records:
        pr_auc, brier = selection_objective(record)
        ranked.append((record, pr_auc, brier))
    eligible = [item for item in ranked if item[1] is not None]
    if not eligible:
        raise ValueError("No candidate has supported PR-AUC values.")

    best_pr_auc = max(item[1] for item in eligible)
    shortlist = [
        item for item in eligible
        if best_pr_auc - item[1] <= config["selection_pr_auc_tolerance"]
    ]
    best_brier = min(item[2] for item in shortlist)
    shortlist = [
        item for item in shortlist
        if item[2] - best_brier <= config["selection_brier_tolerance"]
    ]
    complexity = {"climatology": 0, "lr": 1, "gbt": 2}
    return min(
        shortlist,
        key=lambda item: (
            complexity[item[0]["algorithm"]],
            item[0]["candidate_id"],
        ),
    )[0]


In [ ]:
tuning_records = []
for algorithm in ("lr", "gbt"):
    options = (
        [{"regParam": value} for value in config["lr_reg_params"]]
        if algorithm == "lr"
        else config["gbt_configs"]
    )
    for index, parameters in enumerate(options):
        candidate_id = f"{algorithm}_pilot_{index}"
        fitted = build_pipeline(algorithm, parameters, variant="B").fit(pilot)
        predictions = score_model(fitted, selection).persist(StorageLevel.DISK_ONLY)
        tuning_records.append({
            "candidate_id": candidate_id,
            "algorithm": algorithm,
            "parameters": parameters,
            "metrics": threshold_metrics(predictions, config["thresholds_mm"]),
        })
        predictions.unpersist()
        print(f"Finished {candidate_id}: {parameters}")


In [ ]:
best_parameters = {}
for algorithm in ("lr", "gbt"):
    chosen = choose_candidate([
        record for record in tuning_records if record["algorithm"] == algorithm
    ])
    best_parameters[algorithm] = chosen["parameters"]

display({
    "selection_primary_metric": "mean PR-AUC on supported thresholds",
    "tie_breaker": "mean Brier score within frozen tolerances",
    "classification_cutoff": "fixed at 0.5 for reference F1",
    "best_parameters": best_parameters,
})


In [ ]:
finalists = {}
selection_records = []

baseline_selection = climatology_scores(selection)
baseline_record = {
    "candidate_id": "climatology",
    "algorithm": "climatology",
    "variant": "baseline",
    "metrics": threshold_metrics(baseline_selection, config["thresholds_mm"]),
}
selection_records.append(baseline_record)


In [ ]:
for algorithm in ("lr", "gbt"):
    for variant in ("A", "B"):
        name = f"{algorithm}_{variant}"
        fitted = build_pipeline(algorithm, best_parameters[algorithm], variant).fit(train)
        fitted.write().mode("overwrite").save(f"{MODEL_ROOT}/{name}")
        finalists[name] = fitted

        predictions = score_model(fitted, selection).persist(StorageLevel.DISK_ONLY)
        selection_records.append({
            "candidate_id": name,
            "algorithm": algorithm,
            "variant": variant,
            "parameters": best_parameters[algorithm],
            "metrics": threshold_metrics(predictions, config["thresholds_mm"]),
        })
        predictions.unpersist()


In [ ]:
selection_candidates = [
    record for record in selection_records
    if record["candidate_id"] in {"climatology", "lr_B", "gbt_B"}
]
selected = choose_candidate(selection_candidates)
selected_name = selected["candidate_id"]
selected_selection_scores = (
    baseline_selection
    if selected_name == "climatology"
    else score_model(finalists[selected_name], selection)
)
selection_ece = weighted_ece(selected_selection_scores)

display([
    {
        "model": row["candidate_id"],
        "threshold_mm": metric["threshold_mm"],
        "pr_auc": metric["pr_auc"],
        "brier": metric["brier"],
        "f1_at_0_5": metric["f1"],
    }
    for row in selection_records
    for metric in row["metrics"]
])
print(f"Selected on 2023 selection data: {selected_name}")


In [ ]:
calibration_scores = mark_unseen(
    expand_thresholds(calibration_base), training_stations
)
if selected_name == "climatology":
    calibration_predictions = climatology_scores(calibration_scores)
else:
    calibration_predictions = score_model(
        finalists[selected_name], calibration_scores
    )


In [ ]:
use_calibration = selection_ece > config["calibration_ece_trigger"]
calibrator = None
if use_calibration:
    calibrator = IsotonicRegression(
        featuresCol="p", labelCol="label",
        predictionCol="calibrated_p", isotonic=True,
    ).fit(calibration_predictions)
    calibrator.write().mode("overwrite").save(f"{MODEL_ROOT}/calibrator")


In [ ]:
selection_summary = {
    "selected_model": selected_name,
    "selected_parameters": selected.get("parameters"),
    "primary_thresholds_mm": primary_thresholds,
    "selection_ece": selection_ece,
    "calibration_ece_trigger": config["calibration_ece_trigger"],
    "use_isotonic_calibration": use_calibration,
    "probability_cutoff": REFERENCE_CUTOFF,
    "cutoff_policy": "fixed reference; never optimized",
    "selection_metrics": selection_records,
    "test_year_accessed": False,
    "selected_model_artifact": (
        f"{MODEL_ROOT}/climatology" if selected_name == "climatology"
        else f"{MODEL_ROOT}/{selected_name}"
    ),
    "calibrator_artifact": f"{MODEL_ROOT}/calibrator" if use_calibration else None,
}
selection_json = json.dumps(selection_summary, allow_nan=False)
spark.createDataFrame([(selection_json,)], "value string").coalesce(1).write.mode(
    "overwrite"
).text(f"{RESULTS_ROOT}/selection_summary")
spark.createDataFrame([(selection_json,)], "value string").coalesce(1).write.mode(
    "overwrite"
).text(f"{MODEL_ROOT}/selected_manifest")

print("Selection and optional calibration are complete. The 2024 holdout has not been read.")


## Final holdout

This is the only cell that reads 2024 model inputs. Selection and calibration
choices above are already fixed. Report PR-AUC and Brier as primary metrics;
F1 at 0.5 is a reference metric, and probability reliability is reported separately.


In [ ]:
test_base = base.where("split = 'test' AND label_valid")
test = mark_unseen(expand_thresholds(test_base), training_stations).persist(
    StorageLevel.DISK_ONLY
)
test_count = test.count()
if test_count == 0:
    raise ValueError("The final holdout has no valid threshold-expanded rows.")

test_models = {
    "climatology": None,
    "lr_B": finalists["lr_B"],
    "gbt_B": finalists["gbt_B"],
}
test_scores = {}
evaluation_rows = []


In [ ]:
for name, model in test_models.items():
    predictions = (
        climatology_scores(test)
        if model is None
        else score_model(model, test)
    ).persist(StorageLevel.DISK_ONLY)
    test_scores[name] = predictions
    predictions.write.mode("overwrite").parquet(f"{RESULTS_ROOT}/scores/{name}")

    evaluation_rows.append({
        "model": name,
        "per_threshold": threshold_metrics(predictions, config["thresholds_mm"]),
        "ece": weighted_ece(predictions),
    })
    reliability_table(predictions).write.mode("overwrite").json(
        f"{RESULTS_ROOT}/reliability/{name}"
    )


In [ ]:
if calibrator is not None:
    raw = test_scores[selected_name]
    calibrated = (
        calibrator.transform(raw)
        .withColumnRenamed("p", "raw_p")
        .withColumnRenamed("calibrated_p", "p")
    ).persist(StorageLevel.DISK_ONLY)
    test_scores["selected_calibrated"] = calibrated
    calibrated.write.mode("overwrite").parquet(
        f"{RESULTS_ROOT}/scores/selected_calibrated"
    )
    evaluation_rows.append({
        "model": "selected_calibrated",
        "per_threshold": threshold_metrics(calibrated, config["thresholds_mm"]),
        "ece": weighted_ece(calibrated),
    })
    reliability_table(calibrated).write.mode("overwrite").json(
        f"{RESULTS_ROOT}/reliability/selected_calibrated"
    )


display(evaluation_rows)


In [ ]:
import random

comparison_name = "selected_calibrated" if calibrator is not None else selected_name
selected_test_scores = test_scores[comparison_name]
baseline_test_scores = test_scores["climatology"]
keys = ["station_id", "epoch", "threshold_mm"]
selected_losses = selected_test_scores.select(
    *keys,
    "local_date",
    "label",
    F.pow(F.col("p") - F.col("label"), 2).alias("selected_loss"),
)
baseline_losses = baseline_test_scores.select(
    *keys,
    F.col("label").alias("baseline_label"),
    F.pow(F.col("p") - F.col("label"), 2).alias("baseline_loss"),
)
paired_losses = selected_losses.join(baseline_losses, keys, "inner")
if paired_losses.where("label != baseline_label").limit(1).count():
    raise ValueError("Paired test labels differ.")
if paired_losses.count() != selected_losses.count() or baseline_losses.count() != selected_losses.count():
    raise ValueError("Selected and baseline test cohorts do not match.")


In [ ]:
weekly = (
    paired_losses.withColumn("block", F.date_trunc("week", "local_date"))
    .groupBy("block")
    .agg(
        F.sum(F.col("selected_loss") - F.col("baseline_loss")).alias("difference_sum"),
        F.count("*").alias("n"),
    )
    .orderBy("block")
    .collect()
)


In [ ]:
if len(weekly) > 60:
    raise ValueError("Expected at most 60 calendar-week blocks in the 2024 holdout.")
paired_difference = sum(row["difference_sum"] for row in weekly) / sum(row["n"] for row in weekly)
if len(weekly) < 2:
    paired_ci95 = None
else:
    rng = random.Random(config["sampling_seed"])
    bootstrap = []
    for _ in range(config["bootstrap_replicates"]):
        sample = [weekly[rng.randrange(len(weekly))] for _ in weekly]
        bootstrap.append(
            sum(row["difference_sum"] for row in sample) / sum(row["n"] for row in sample)
        )
    bootstrap.sort()
    paired_ci95 = [
        bootstrap[int(0.025 * (len(bootstrap) - 1))],
        bootstrap[int(0.975 * (len(bootstrap) - 1))],
    ]
paired_weekly_brier = {
    "comparison": "selected model minus climatology baseline",
    "difference": paired_difference,
    "ci95": paired_ci95,
    "week_blocks": len(weekly),
    "bootstrap_replicates": config["bootstrap_replicates"] if paired_ci95 else 0,
}
display(paired_weekly_brier)


In [ ]:
def monotonicity_summary(scores):
    ordered = Window.partitionBy("station_id", "epoch").orderBy("threshold_mm")
    differences = (
        scores.withColumn("previous_p", F.lag("p").over(ordered))
        .where("previous_p IS NOT NULL")
        .withColumn("increase", F.col("p") - F.col("previous_p"))
    )
    return differences.agg(
        F.count("*").alias("threshold_pairs"),
        F.avg(
            (F.col("increase") > config["monotonicity_tolerance"]).cast("double")
        ).alias("violation_rate"),
        F.max(F.greatest(F.col("increase"), F.lit(0))).alias("max_increase"),
    ).first().asDict()


In [ ]:
for row in evaluation_rows:
    name = row["model"]
    row["monotonicity"] = monotonicity_summary(test_scores[name])

unseen_test = mark_unseen(
    expand_thresholds(test_base, config["unseen_thresholds_mm"]),
    training_stations,
)
if selected_name == "climatology":
    unseen_scores = climatology_scores(unseen_test)
else:
    unseen_scores = score_model(finalists[selected_name], unseen_test)
if calibrator is not None:
    unseen_scores = (
        calibrator.transform(unseen_scores)
        .withColumnRenamed("p", "raw_p")
        .withColumnRenamed("calibrated_p", "p")
    )

unseen_metrics = threshold_metrics(unseen_scores, config["unseen_thresholds_mm"])
unseen_summary = {
    "model": "selected_calibrated" if calibrator is not None else selected_name,
    "per_threshold": unseen_metrics,
    "monotonicity": monotonicity_summary(unseen_scores),
}
display(unseen_summary)


In [ ]:
summary = {
    "selected_model": selected_name,
    "calibrated": use_calibration,
    "test_rows": test_count,
    "classification_cutoff_policy": "F1 at 0.5 is descriptive; cutoff was not optimized",
    "metrics": evaluation_rows,
    "unseen_thresholds": unseen_summary,
    "post_test_tuning": False,
}
spark.createDataFrame(
    [(json.dumps(summary, allow_nan=False),)], "value string"
).coalesce(1).write.mode("overwrite").text(f"{RESULTS_ROOT}/evaluation_summary")



In [ ]:
def precision_recall_points(scores):
    bins = config["auc_num_bins"]
    grouped = (
        scores.withColumn("score_bin", F.floor(F.col("p") * bins).cast("int"))
        .groupBy("score_bin")
        .agg(F.sum("label").alias("positives"), F.count("*").alias("n"))
    )
    total_positives = grouped.agg(F.sum("positives")).first()[0]
    ordered = Window.orderBy(F.desc("score_bin")).rowsBetween(
        Window.unboundedPreceding, 0
    )
    return (
        grouped.withColumn("tp", F.sum("positives").over(ordered))
        .withColumn("predicted_positive", F.sum("n").over(ordered))
        .withColumn("precision", F.col("tp") / F.col("predicted_positive"))
        .withColumn("recall", F.col("tp") / F.lit(total_positives))
        .orderBy("recall")
        .toPandas()
    )


In [ ]:
primary_threshold = primary_thresholds[0]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, scores in test_scores.items():
    at_threshold = scores.where(F.col("threshold_mm") == primary_threshold)
    reliability = reliability_table(at_threshold).orderBy("bin").toPandas()
    axes[0].plot(
        reliability["predicted_probability"],
        reliability["observed_frequency"],
        marker="o",
        label=name,
    )
    curve = precision_recall_points(at_threshold)
    axes[1].plot(curve["recall"], curve["precision"], label=name)

axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set(xlabel="Mean predicted probability", ylabel="Observed frequency")
axes[0].set_title(f"Reliability at {primary_threshold:g} mm")
axes[1].set(xlabel="Recall", ylabel="Precision")
axes[1].set_title(f"Precision–recall at {primary_threshold:g} mm")
for axis in axes:
    axis.legend(fontsize=8)
fig.tight_layout()
plt.show()
plt.close(fig)
print(f"Metrics and scores saved under {RESULTS_ROOT}")

for frame in [pilot, train, selection, training_stations, test, *test_scores.values()]:
    frame.unpersist()
